In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
import os
import time
import requests
import pandas as pd
import datetime
from dotenv import load_dotenv

load_dotenv('.env')

# 설정
SERVICE_KEY = os.getenv("OPENWEATHER_API_KEY")
BASE_DIR = "data" #데이터를 저장할 기본 디렉토리를 "data"로 설정
os.makedirs(BASE_DIR, exist_ok=True) #data 폴더가 없으면 생성하고, 있으면 무시함

# 공통 설정
STN_ID = 108 #기상 관측소 ID(서울지점)
NUM_OF_ROWS = 800 #한 번 호출시 가져올 최대 데이터 개수
START_DATE = datetime.date(2000, 1, 10) #수집 시작 날짜
END_DATE = datetime.date.today() - datetime.timedelta(days=1) #수집 종료 날짜(어제까지)

# 1. 시간별 관측자료 수집 (ASOS)
def get_hourly_data():
    """시간별 데이터 수집"""
    CSV_FILE = os.path.join(BASE_DIR, 'asos_seoul_hourly.csv') #시간별 데이터를 저장할 CSV 파일 경로 설정
    API_URL = "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"
    
    def load_existing_times(): #이미 수집된 시간 데이터를 확인하는 내부 함수 
        """기존에 수집된 시간 데이터 로드"""
        if not os.path.exists(CSV_FILE): #csv파일 존재시 빈 set반환(처음)
            return set()
        try:
            df = pd.read_csv(CSV_FILE)
            return set(df['tm'].astype(str)) #시간(tm)컬럼을 문자열로 변환하여 set로 반환
        except Exception as e:
            print(f"기존 데이터 로드 실패: {e}")
            return set()

    def fetch_day_data(date_obj, existing_times): #하루치 데이터 수집 함수
        """지정된 하루치(00~23시) 데이터를 수집"""
        all_results = []
        start_dt = datetime.datetime.combine(date_obj, datetime.time(0, 0)) #시작 시간
        end_dt = datetime.datetime.combine(date_obj, datetime.time(23, 0)) #종료 시간

        page = 1 #페이지 번호
        while True: #모든 페이지를 가져올때까지 반복
            params = { #API 호출에 필요한 파라미터들
                'serviceKey': SERVICE_KEY,
                'numOfRows': NUM_OF_ROWS,
                'pageNo': page,
                'dataCd': 'ASOS',
                'dateCd': 'HR',
                'startDt': start_dt.strftime("%Y%m%d"),
                'startHh': "00",
                'endDt': end_dt.strftime("%Y%m%d"),
                'endHh': "23",
                'stnIds': STN_ID,
                'dataType': 'JSON'
            }

            try:
                response = requests.get(API_URL, params=params, timeout=60) #API에 GET 요청을 보내고 60초 타임아웃 설정
                response.raise_for_status() #HTTP 오류가 있으면 예외 발생
            except requests.exceptions.RequestException as e:
                print(f"[{date_obj}] 요청 실패: {e}")
                return []

            try: #JSON 파싱 예외 처리 시작
                data = response.json() #응답을 JSON으로 파싱
                result_code = data['response']['header']['resultCode'] #API 응답 코드 추출
                result_msg = data['response']['header']['resultMsg'] #API 응답 메시지 추출
                if result_code != '00': #응답 코드가 성공(00)이 아니면 예외 발생
                    print(f"[{date_obj}] API 오류: {result_msg}")
                    return []
            except Exception as e:
                print(f"[{date_obj}] JSON 파싱 실패: {e}")
                return []

            items = data['response']['body']['items'].get('item', [])
            if not items:
                break #데이터 항목이 없으면 while 종료

            # 새로운 데이터만 추가
            for item in items: #해당 시간이 이미 수집되지 않았다면 all_result에 추가
                if item['tm'] not in existing_times:
                    all_results.append(item)

            if len(items) < NUM_OF_ROWS: #가져온 항목 수가 최대치보다 적으면 (마지막 페이지) while종료
                break
            page += 1 #페이지 증가

        print(f"[{date_obj}] {len(all_results)}건 수집 성공")
        return all_results

    def append_to_csv(new_data):
        """새로운 데이터를 CSV에 추가"""
        if not new_data:
            return
        
        df = pd.DataFrame(new_data) #새 데이터 데이터프레임으로 불러오기
        
        # 기존 파일과 동일한 컬럼 순서로 정렬 (있는 컬럼만)
        target_columns = ['tm', 'ta', 'rn', 'ws', 'wd', 'hm', 'pa', 'ps', 'td', 'pv']
        available_columns = [col for col in target_columns if col in df.columns] #target_columns중에 df에 있는 컬럼만 추출
        df = df[available_columns] #추출한 컬럼만 사용
        
        # 수치형 컬럼 변환 (tm 제외)
        for col in available_columns:
            if col != 'tm': #시간 컬럼이 아니면 수치형으로 변환
                df[col] = pd.to_numeric(df[col], errors='coerce')
            
        write_header = not os.path.exists(CSV_FILE) #파일이 없으면 헤더를 쓰고, 있으면 헤더 생략
        df.to_csv(CSV_FILE, mode='a', header=write_header, index=False) #파일에 데이터 추가 후 저장

    def get_last_collected_date():
        """마지막으로 수집된 날짜 확인"""
        if not os.path.exists(CSV_FILE):#CSV 파일이 없으면 시작 날짜 반환
            return START_DATE
        try:
            df = pd.read_csv(CSV_FILE)
            last_time = pd.to_datetime(df['tm']).max() #가장 최근 시간 찾기
            return last_time.date() + datetime.timedelta(days=1) #마지막 날짜의 다음 날 반환
        except Exception as e:
            print(f"마지막 수집일 확인 실패: {e}")
            return START_DATE

#실행 부분
    # 데이터 수집 실행
    print("시간별 데이터 수집 시작...")
    existing_times = load_existing_times() #기존 수집된 시간 데이터 로드
    current_date = get_last_collected_date() #수집을 시작할 날짜 확인
    
    print(f"현재 수집된 데이터: {len(existing_times):,}개 시간대")
    print(f"수집 시작 날짜: {current_date}")
    print(f"수집 종료 날짜: {END_DATE}")

    if current_date > END_DATE: #시작 날짜가 종료 날짜보다 늦으면 완료
        print("이미 최신 데이터까지 모두 수집되어 있습니다!")
        return

    collected_count = 0
    while current_date <= END_DATE: #시작 날짜가 종료 날짜보다 빠르면 종료 날짜까지 반복
        new_data = fetch_day_data(current_date, existing_times)
        if new_data:
            append_to_csv(new_data)
            collected_count += len(new_data)
            
            # 새로 수집한 데이터의 시간을 기존 set에 추가
            for item in new_data:
                existing_times.add(item['tm'])
            
        current_date += datetime.timedelta(days=1)
        time.sleep(0.5)  # API 과부하 방지

    if collected_count == 0:
        print("새로운 시간별 데이터가 없습니다. 이미 최신 상태입니다!")
    else:
        print(f"시간별 데이터 수집 완료! (새로 추가된 데이터: {collected_count:,}건)")